# De Bruijn graphs



## Eulerian walk



```
eulerian_walk:
Beginning at first_node as node

For node:
    follow a random valid edge from node
    remove edge
    recurse
```


In [1]:
import gzip

def iter_fastq_seqs_gz(path):
    # function to open the gzipped fastq file and create a generator object
    # that allows you to iterate over the sequences in the fastq without saving
    # them all to memory at once
    with gzip.open(path, "rt", encoding="utf-8", newline="") as fh:
        i=0
         
        while True:
            i +=1
            header = fh.readline()
        
            if not header:
                return  # EOF
            # if i > 100000000:
            #     return 

            print(f'\r[build] progress: {i} reads', end='', flush=True)

            seq = fh.readline()
            plus = fh.readline()
            qual = fh.readline()

            yield seq.strip()

In [4]:
"""De Bruijn Graph Genome Assembly Module.

This module provides classes and functions for constructing De Bruijn graphs
from sequencing reads and performing genome assembly using Eulerian path
traversal.
"""

from collections import defaultdict
import random


def _eulerian_walk(graph_to_consume, startnode):
        # iterative version of the Eulerian walk using a stack to avoid recursion error

        stack = [startnode]
        path = []

        while stack:
            currnode = stack[-1]
            if graph_to_consume[currnode]:
                # go to the next node along one of the edges
                nextnode = graph_to_consume[currnode].pop()
                stack.append(nextnode)
            else:
                # if there are no edges to go to from the current node, start emptying the stack into the path
                path.append(stack.pop())

        # reverse the list of nodes we've been to because they're added in reverse order (last-in, first-out)
        path.reverse()
        return path

def tour_to_sequence(nodes):
    """Convert a tour of (k-1)-mers into a DNA sequence.

    Args:
        nodes (list): List of (k-1)-mer strings in order.

    Returns:
        str: Assembled DNA sequence.

    Example:
        >>> dbg = DeBruijnGraph([], k=4)
        >>> tour = ['ATG', 'TGG', 'GGC', 'GCG']
        >>> tour_to_sequence(tour)
        'ATGGCG'
    """
    if not nodes:
        return ""

    return nodes[0] + "".join(n[-1] for n in nodes[1:])

def _get_n50(contig_lengths: list[int]) -> int:
    """
    Take a list of contigs' lengths, sorted from greatest to least, and calculate the shortest contig for which
    all contigs of that length or longer, when put together, represent 50% or more of the assembly

    @param contig_lengths: list of ints representing the lengths of contigs, MUST be sorted from highest to lowest
    @return: int, the n50 statistic
    """
    total_length = sum(contig_lengths)

    # initialize a value to track the total length covered by the reads we're about to iterate over
    dist = 0

    for curr_length in contig_lengths:
        # add the current contig's length to our sum
        dist += curr_length
        # check if our sum is >= half of the total length of the contigs
        if dist >= total_length / 2:
            # quit out because we've hit the shortest contig that meets our conditions
            return curr_length


class DeBruijnGraph:
    """Main class for De Bruijn graphs and genome assembly.

    This class builds De Bruijn graphs from sequencing reads and performs
    genome assembly by finding Eulerian paths through connected components
    of the graph.

    Attributes:
        graph (defaultdict): Adjacency list representation of graph edges.
            Keys are (k-1)-mers, values are lists of adjacent (k-1)-mers.
        k (int): The k-mer size used for graph construction.

    Example:
        >>> reads = ["ATGGCGTACG", "GCGTACGTTA", "ACGTTACCAT"]
        >>> dbg = DeBruijnGraph(reads, k=6)
        >>> contigs = dbg.assemble_contigs(seed=42)
        >>> len(contigs) > 0
        True
    """

    def __init__(self, reads, k, ignore_ambiguous=True, verbose=True):
        """Initialize De Bruijn graph from sequencing reads.

        Args:
            reads (list): List of DNA sequence strings.
            k (int): K-mer size for graph construction.

        Example:
            >>> reads = ["ATGGCG", "GCGTGC", "TGCAAC"]
            >>> dbg = DeBruijnGraph(reads, k=4)
            >>> len(dbg.graph) > 0
            True
        """
        self.graph = defaultdict(list)
        self.k = k
        self.ignore_ambiguous = ignore_ambiguous

        self.balance = defaultdict(int)

        self.startnodes = []
        self.stats = {}
        self.contig_lists = None  # list of node-path lists created by .assemble_contigs()
        self.build_graph_from_reads(reads, verbose=verbose)

        self.build_graph_from_reads(reads, k)

    def add_edge(self, prefix, suffix):
        """Add a directed edge to the graph.

        Args:
            left (str): Source (k-1)-mer node.
            right (str): Destination (k-1)-mer node.

        Example:
            >>> dbg = DeBruijnGraph([], k=4)
            >>> dbg.add_edge("ATG", "TGG")
            >>> "TGG" in dbg.graph["ATG"]
            True
        """
        self.graph[prefix].append(suffix)

    def remove_edge(self, prefix):
        """Remove a directed edge from the graph.
           Removed edge is random as long as edges were shuffled during graph creation.

        Args:
            prefix (str): Source (k-1)-mer node.

        Example:
            >>> dbg = DeBruijnGraph([], k=4)
            >>> dbg.add_edge("ATG", "TGG")
            >>> dbg.remove_edge("ATG", "TGG")
            >>> len(dbg.graph["ATG"])
            0
        """
        suffix = self.graph[prefix].pop()
        return suffix

    def build_graph_from_reads(self, reads, verbose = True):
        """Build De Bruijn graph from multiple sequencing reads.

        Extracts all k-mers from all reads and adds edges between
        consecutive (k-1)-mers within each k-mer.

        Args:
            reads (list): List of DNA sequence strings.
            verbose (bool): indicates if information should be printed to terminal about graph construction

        Example:
            >>> reads = ["ATGGC", "TGGCA"]
            >>> dbg = DeBruijnGraph([], k=4)
            >>> dbg.build_graph_from_reads(reads, 4)
            >>> "ATG" in dbg.graph
            True
        """
        reads_total = 0
        bases_total = 0
        ignored_short = 0
        ignored_amb = 0
        kmers_added = 0

        for read in reads:
            read = read.strip().upper()
            reads_total += 1
            bases_total += len(read)

            if len(read) < self.k:
                ignored_short += 1
                continue # escapes read's loop, ignoring it because it's too short

            if self.ignore_ambiguous and any(b not in "ACGT" for b in read):
                ignored_amb += 1
                continue # escapes read's loop, ignoring it because there are ambiguous nucleotides in the read

            for i in range(len(read) - self.k + 1):
                # loops through kmers of read
                kmer = read[i : i + self.k]

                prefix = kmer[:-1]
                suffix = kmer[1:]

                # add an entry into the graph indicating that the current suffix's node  is connected to the current prefix's node via an edge
                self.graph[prefix].append(suffix)

                self.balance[prefix] += 1 # indicates another edge leaves this prefix's node
                self.balance[suffix] -= 1 # indicates another edge enters this prefix's node

                kmers_added += 1

        # make a list of nodes (k-1mers) for which they have more edges entering them than leaving
        self.startnodes = [n for (n, b) in self.balance.items() if b > 0]

        edges = sum(len(adj) for adj in self.graph.values())
        nodes = len(set(self.balance.keys()) | set(self.graph.keys()))
        bal_pos = sum(1 for b in self.balance.values() if b > 0) # number of nodes with more edges going out than coming in
        bal_neg = sum(1 for b in self.balance.values() if b < 0) # number of nodes with more edges coming in than going out
        bal_zero = sum(1 for b in self.balance.values() if b == 0) # number of balanced nodes
        bal_abs_gt1 = sum(1 for b in self.balance.values() if abs(b) > 1) # number of nodes unbalanced by more than 1 edge

        self.stats = {
            "reads_total": reads_total,
            "bases_total": bases_total,
            "reads_ignored_too_short": ignored_short,
            "reads_ignored_ambiguous": ignored_amb,
            "kmers_added": kmers_added,
            "nodes": nodes,
            "edges": edges,
            "balance_pos": bal_pos,
            "balance_neg": bal_neg,
            "balance_zero": bal_zero,
            "balance_abs_gt1": bal_abs_gt1,
            "start_nodes": len(self.startnodes),
        }

        if verbose:
            ignored = ignored_short + ignored_amb
            pct_ignored = (ignored / reads_total * 100.0) if reads_total else 0.0
            print(f"[build] k={self.k}")
            print(f"[build] received {reads_total} reads; ignored {ignored} ({pct_ignored:.2f}%)")
            if ignored_short:
                print(f"[build] ignored because too short={ignored_short}")
            if ignored_amb:
                print(f"[build] ignored because ambiguous base(s)={ignored_amb}")
            print(f"[build] {nodes} nodes; {edges} edges ({kmers_added} kmers)")
            print(f"""[build] balance: +={bal_pos} -={bal_neg}; {bal_zero} 0s; |abs(b)|>1={bal_abs_gt1}""")
            if bal_abs_gt1:
                print("[build] warning: some nodes have absolute balance > 1")
            if not self.startnodes and edges:
                print("[build] warning: no balance > 0 start nodes detected; assembly will rely on leftover cleanup step.")



    def eulerian_walk_recursive(self, node, graph, seed=None):
        if seed is not None:
            random.seed(seed)

        tour = []

        def _walk(current_node):
            while graph[current_node]:
                if seed is not None:
                    next_node = random.choice(graph[current_node])
                    graph[current_node].remove(next_node)   # remove ONE edge
                else:
                    next_node = graph[current_node].pop()   # deterministic + removes edge
                _walk(next_node)
            tour.append(current_node)

        _walk(node)
        return tour  # reverse order


    def assemble_contigs(self, seed=None, verbose=True):
        """Assemble all contigs from the De Bruijn graph.

        Finds all connected components and generates an Eulerian path
        for each component, producing multiple assembled contigs.

        Args:
            seed (int, optional): Random seed for reproducible assembly.

        Returns:
            list: List of assembled contig sequences (DNA strings).

        Example:
            >>> reads = ["ATGGCGTACG", "GCGTACGTTA", "ACGTTACCAT"]
            >>> dbg = DeBruijnGraph(reads, k=6)
            >>> contigs = dbg.assemble_contigs(seed=42)
            >>> all(isinstance(c, str) for c in contigs)
            True
        """
        rng = random.Random(seed) # local rng

        # make a deep copy of the graph so we can remove elements for our walk without editing the original graph object
        working_graph = defaultdict(list, {currentnode:nextnodes_list.copy() for currentnode, nextnodes_list in self.graph.items()})

        for currentnode in working_graph:
            rng.shuffle(working_graph[currentnode]) # shuffle edge lists

        starts = list(self.startnodes)
        rng.shuffle(starts) # shuffle starts

        contigs = []

        # prioritize balance-based start candidates
        for s in starts:
            path_nodes_list = _eulerian_walk(working_graph, s)
            contigs.append(path_nodes_list)

        # exhaust leftovers
        # there's gotta be a prettier way to do this
        while True:
            next_start = None

            for currentnode, nextnodes_list in working_graph.items():
                # are there any available currentnodes to make more contigs?
                if nextnodes_list: 
                    next_start = currentnode
                    break

            if next_start is None:
                # didn't find any
                break

            path_nodes_list = _eulerian_walk(working_graph, next_start) # make another contig
            contigs.append(path_nodes_list)

        self.contig_lists = contigs

        # assembly diagnostics
        # add actual stats after this is just for debugging the assembly itself
        leftover_edges = sum(len(adj) for adj in working_graph.values())
        contig_lengths = [len(tour_to_sequence(p)) for p in contigs]
        contig_lengths.sort(reverse=True)

        self.stats.update({
            "assembly_seed": seed,
            "num_contigs": len(contigs),
            "total_length": sum(contig_lengths),
            "max_bp": contig_lengths[0] if contig_lengths else 0,
            "leftover_edges_after_exhaust": leftover_edges,
            "contig_lengths": contig_lengths
        })

        self.stats.update({
            "longest_contig": contig_lengths[0] if contig_lengths else 0,
            "shortest_contig": contig_lengths[-1] if contig_lengths else 0,
            "mean_length": self.stats['total_length']/self.stats['num_contigs'],
            "n50": _get_n50(contig_lengths) if contig_lengths else 0
            })

        if verbose:
            print(f"[assembly] seed = {seed}; contigs = {self.stats['num_contigs']}")
            print(f"[assembly] leftover edges = {leftover_edges}")
            print()

        return contigs

    def get_assembly_stats(self):
        """Access the relevant assembly statistics for assembled contigs.

        Args:
            self.

        Returns:
            dict: Dictionary containing assembly statistics:
                - num_contigs: Total number of contigs
                - total_length: Total assembled sequence length
                - longest_contig: Length of longest contig
                - shortest_contig: Length of shortest contig
                - mean_length: Mean contig length
                - n50: N50 statistic

        Example:
            >>> contigs = ["ATGGCG", "TTTAAA", "CCCCCCCCCC"]
            >>> dbg = DeBruijnGraph([], k=4)
            >>> stats = dbg.get_assembly_stats(contigs)
            >>> stats['num_contigs']
            3
        """

        return {
            "num_contigs": self.stats["num_contigs"],
            "total_length": self.stats["total_length"],
            "longest_contig": self.stats["longest_contig"],
            "shortest_contig": self.stats["shortest_contig"],
            "mean_length": self.stats["mean_length"],
            "n50": self.stats["n50"],
        }

    def write_fasta(self, out_path, sort_by_len=True, wrap=60):
        """Write assembled contigs to a FASTA file.

        Args:
            contigs (list): List of contig sequences.
            filename (str): Output FASTA filename.

        Example:
            >>> contigs = ["ATGGCG", "TTTAAA"]
            >>> dbg = DeBruijnGraph([], k=4)
            >>> dbg.write_fasta("output.fasta")
        """
        if self.contig_lists is None:
            raise RuntimeError("assemble_contigs() has not been run yet.")
        contigs = []

        for i, path_nodes in enumerate(self.contig_lists, start=1):
            # convert the path from a list of nodes to the actual sequence it represents
            seq = tour_to_sequence(path_nodes)
            contigs.append((i, seq))

        if sort_by_len:
            contigs.sort(key=lambda x: len(x[1]), reverse=True)

        with open(out_path, "w", encoding="utf-8") as f:
            for idx, seq in contigs:
                f.write(f">contig_{idx:06d} len={len(seq)} k={self.k}\n")
                for i in range(0, len(seq), wrap):
                    f.write(seq[i:i+wrap] + "\n")


---
# Toy Example

In [5]:
print("="*60)
print("EXAMPLE 1: Assembling 9 overlapping reads into one contig")
print("="*60)

toy_reads_1 = [
    "ATGGCGTACG",  # Read 1
    "GGCGTACGTT",  # Read 2: overlaps with Read 1
    "CGTACGTTAC",  # Read 3: overlaps with Read 2
    "TACGTTACCA",  # Read 4: overlaps with Read 3
    "CGTTACCATG",  # Read 5: overlaps with Read 4
    "TTACCATGGG",  # Read 6: overlaps with Read 5
    "ACCATGGGCC",  # Read 7: overlaps with Read 6
    "CATGGGCCTA",  # Read 8: overlaps with Read 7
    "TGGGCCTAAA"   # Read 9: overlaps with Read 8
]

print(f"\nInput: {len(toy_reads_1)} reads")
print("First read:  ", toy_reads_1[0])
print("Last read:   ", toy_reads_1[-1])
k=6
print(f"\nBuilding De Bruijn graph with k=6...")

# 1) Build the graph
dbg = DeBruijnGraph(toy_reads_1, k=k)

# Optional: quick sanity checks
num_nodes = len(dbg.graph)
num_edges = sum(len(v) for v in dbg.graph.values())
print(f"Graph has {num_nodes} nodes and {num_edges} edges")

# 2) Assemble contigs
print("\nAssembling contigs...")
contigs = dbg.assemble_contigs(seed=42)

print(f"\nAssembled {len(contigs)} contig(s):")
for i, c in enumerate(contigs, start=1):
    print(f"  Contig {i}: length={len(c)}")
    print(f"    {c}")

# 3) Stats
stats = dbg.get_assembly_stats()
print("\nAssembly stats:")
for k_stat, v in stats.items():
    print(f"  {k_stat}: {v}")

# 4) (Optional) write to fasta
dbg.write_fasta("toy_example_1_contigs.fasta")
print("\nWrote FASTA to toy_example_1_contigs.fasta")

EXAMPLE 1: Assembling 9 overlapping reads into one contig

Input: 9 reads
First read:   ATGGCGTACG
Last read:    TGGGCCTAAA

Building De Bruijn graph with k=6...
[build] k=6
[build] received 9 reads; ignored 0 (0.00%)
[build] 22 nodes; 45 edges (45 kmers)
[build] balance: +=9 -=9; 4 0s; |abs(b)|>1=0
[build] k=6
[build] received 9 reads; ignored 0 (0.00%)
[build] 22 nodes; 90 edges (45 kmers)
[build] balance: +=9 -=9; 4 0s; |abs(b)|>1=18
[build] warning: some nodes have absolute balance > 1
Graph has 21 nodes and 90 edges

Assembling contigs...
[assembly] seed = 42; contigs = 9
[assembly] leftover edges = 0


Assembled 9 contig(s):
  Contig 1: length=91
    ['ATGGC', 'TGGCG', 'TGGCG', 'GGCGT', 'GGCGT', 'GCGTA', 'GCGTA', 'GCGTA', 'GCGTA', 'CGTAC', 'CGTAC', 'CGTAC', 'CGTAC', 'GTACG', 'GTACG', 'GTACG', 'GTACG', 'GTACG', 'GTACG', 'TACGT', 'TACGT', 'TACGT', 'TACGT', 'ACGTT', 'ACGTT', 'ACGTT', 'ACGTT', 'ACGTT', 'ACGTT', 'CGTTA', 'CGTTA', 'CGTTA', 'CGTTA', 'GTTAC', 'GTTAC', 'GTTAC', 'GTTAC', '

---
# Real data
This section utilizes a FASTQ file with 10 million "perfect" 150 bp reads simulated
from the mouse genome (`GRCm39`) and tasks your program to ingest, assemble, and traverse
the genome with statistical output report.

In [7]:
"""Driver program for mouse genome assembly using De Bruijn graphs.

This script demonstrates how to use the completed DeBruijnGraph class to
assemble 10 million perfect 150bp single-end reads from the mouse genome.

Usage:
    Run all cells in order in a Jupyter notebook.

Expected input:
    - File: mouse_SE_150bp.fq
    - Format: FASTQ
    - Reads: 10 million perfect 150bp single-end reads
    - Source: Simulated from mouse genome using wgsim

Output:
    - mouse_assembly.fasta: Assembled contigs
    - mouse_assembly_stats.txt: Detailed statistics
"""

import time
from datetime import datetime


def write_statistics_file(
    stats_file,
    input_file,
    num_reads,
    avg_read_length,
    k_mer_size,
    random_seed,
    num_nodes,
    num_edges,
    stats,
    contig_lengths,
    timing,
    coverage_estimate,
    assembly_fraction
):
    """Write comprehensive assembly statistics to text file.
    
    Args:
        stats_file (str): Output filename.
        input_file (str): Input FASTQ filename.
        num_reads (int): Number of reads processed.
        avg_read_length (float): Average read length.
        k_mer_size (int): K-mer size used.
        random_seed (int): Random seed used.
        num_nodes (int): Number of graph nodes.
        num_edges (int): Number of graph edges.
        stats (dict): Assembly statistics.
        contig_lengths (list): Sorted list of contig lengths.
        timing (dict): Timing information.
        coverage_estimate (float): Estimated sequencing coverage.
        assembly_fraction (float): Assembly size as % of genome.
    """
    with open(stats_file, 'w') as f:
        f.write("="*80 + "\n")
        f.write("MOUSE GENOME ASSEMBLY STATISTICS\n")
        f.write("="*80 + "\n\n")
        
        f.write(f"Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Input File: {input_file}\n")
        f.write(f"K-mer Size: {k_mer_size}\n")
        f.write(f"Random Seed: {random_seed}\n\n")
        
        f.write("-"*80 + "\n")
        f.write("INPUT DATA\n")
        f.write("-"*80 + "\n")
        f.write(f"Number of reads:         {num_reads:,}\n")
        f.write(f"Average read length:     {avg_read_length:.1f} bp\n")
        f.write(f"Total sequencing data:   {num_reads * avg_read_length:,.0f} bp\n")
        f.write(f"Estimated coverage:      {coverage_estimate:.1f}x\n")
        f.write(f"Read time:               {timing['read_time']:.2f} seconds\n\n")
        
        f.write("-"*80 + "\n")
        f.write("DE BRUIJN GRAPH CONSTRUCTION\n")
        f.write("-"*80 + "\n")
        f.write(f"Graph nodes:             {num_nodes:,}\n")
        f.write(f"Graph edges:             {num_edges:,}\n")
        f.write(f"Average out-degree:      {num_edges/num_nodes:.2f}\n")
        f.write(f"Construction time:       {timing['graph_time']:.2f} seconds\n\n")
        
        f.write("-"*80 + "\n")
        f.write("ASSEMBLY RESULTS\n")
        f.write("-"*80 + "\n")
        f.write(f"Number of contigs:       {stats['num_contigs']:,}\n")
        f.write(f"Total assembly length:   {stats['total_length']:,} bp\n")
        f.write(f"Assembly vs. genome:     {assembly_fraction:.2f}%\n")
        f.write(f"Longest contig:          {stats['longest_contig']:,} bp\n")
        f.write(f"Shortest contig:         {stats['shortest_contig']:,} bp\n")
        f.write(f"Mean contig length:      {stats['mean_length']:,.1f} bp\n")
        f.write(f"N50:                     {stats['n50']:,} bp\n")
        f.write(f"Assembly time:           {timing['assembly_time']:.2f} seconds\n\n")
        
        f.write("-"*80 + "\n")
        f.write("TOP 20 LONGEST CONTIGS\n")
        f.write("-"*80 + "\n")
        for i, length in enumerate(contig_lengths[:20], 1):
            f.write(f"{i:3d}. {length:10,} bp\n")
        f.write("\n")
        
        f.write("-"*80 + "\n")
        f.write("CONTIG LENGTH DISTRIBUTION\n")
        f.write("-"*80 + "\n")
        bins = [
            (">100kb", sum(1 for x in contig_lengths if x > 100000)),
            (">50kb", sum(1 for x in contig_lengths if x > 50000)),
            (">10kb", sum(1 for x in contig_lengths if x > 10000)),
            (">5kb", sum(1 for x in contig_lengths if x > 5000)),
            (">1kb", sum(1 for x in contig_lengths if x > 1000)),
            (">500bp", sum(1 for x in contig_lengths if x > 500)),
        ]
        for bin_name, count in bins:
            f.write(f"Contigs {bin_name:8s}:     {count:,}\n")
        f.write("\n")
        
        f.write("-"*80 + "\n")
        f.write("TIMING SUMMARY\n")
        f.write("-"*80 + "\n")
        total_time = timing['total_time']
        f.write(f"Read time:               {timing['read_time']:8.2f} seconds "
                f"({timing['read_time']/total_time*100:5.1f}%)\n")
        f.write(f"Graph construction:      {timing['graph_time']:8.2f} seconds "
                f"({timing['graph_time']/total_time*100:5.1f}%)\n")
        f.write(f"Assembly:                {timing['assembly_time']:8.2f} seconds "
                f"({timing['assembly_time']/total_time*100:5.1f}%)\n")
        f.write(f"Total time:              {total_time:8.2f} seconds "
                f"({total_time/60:.2f} minutes)\n\n")
        
        f.write("="*80 + "\n")
        f.write("END OF REPORT\n")
        f.write("="*80 + "\n")


def assemble_mouse_genome(
    input_file="mouse_SE_150bp.fq.gz",
    output_fasta="mouse_assembly.fasta", 
    stats_file="mouse_assembly_stats.txt",
    k_mer_size = 51,
    random_seed = None
):
    """Main driver function for mouse genome assembly.
    
    This function orchestrates the complete assembly pipeline:
    1. Load FASTQ reads
    2. Build De Bruijn graph
    3. Assemble contigs
    4. Calculate statistics
    5. Write output files
    
    Returns:
        dict: Dictionary containing assembly results including:
            - dbg: DeBruijnGraph object
            - contigs: List of assembled sequences
            - stats: Assembly statistics dictionary
            - timing: Performance timing information
    """
    print("="*80)
    print("MOUSE GENOME ASSEMBLY PIPELINE")
    print("="*80)
    print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print()
    
    timing = {}
    
    print("STEP 1: Loading sequencing reads")
    print("-"*80)
    print(f"Input file: {input_file}")
    print(f"Expected: 10 million reads, 150bp each")
    print()
    
    start_time = time.time()
    reads = iter_fastq_seqs_gz(input_file)
    timing['read_time'] = time.time() - start_time
    print(f"Time elapsed: {timing['read_time']:.2f} seconds")
    
    print("STEP 2: Building De Bruijn graph")
    print("-"*80)
    print(f"K-mer size: {k_mer_size}")
    # print(f"Building graph from {num_reads:,} reads...")
    print()
    
    start_time = time.time()
    dbg = DeBruijnGraph(reads, k=k_mer_size)
    timing['graph_time'] = time.time() - start_time

    num_reads = dbg.stats["reads_total"]
    total_bases = dbg.stats["bases_total"]
    avg_read_length = total_bases / num_reads if num_reads > 0 else 0

    print(f"Reads loaded: {num_reads:,}")
    print(f"Total bases: {total_bases:,} bp")
    print(f"Average read length: {avg_read_length:.1f} bp")
    
    if num_reads > 0:
        print(f"\nSample reads:")
        print(f"  First: {reads[0][:80]}...")
        print(f"  Last:  {reads[-1][:80]}...")
    print()
    
    # Calculate graph statistics
    num_nodes = dbg.stats["nodes"]
    num_edges = dbg.stats["edges"]
    avg_degree = num_edges / num_nodes if num_nodes > 0 else 0
    
    print(f"Graph construction complete!")
    print(f"  Nodes (unique {k_mer_size-1}-mers): {num_nodes:,}")
    print(f"  Edges (k-mer transitions): {num_edges:,}")
    print(f"  Average out-degree: {avg_degree:.2f}")
    print(f"Time elapsed: {timing['graph_time']:.2f} seconds")
    print()
    
    print("STEP 3: Assembling contigs")
    print("-"*80)
    print(f"Finding connected components and traversing graph...")
    print(f"Random seed: {random_seed} (for reproducibility)")
    print()
    
    start_time = time.time()
    contigs = dbg.assemble_contigs(seed=random_seed) # this is a list of lists of nodes
    timing['assembly_time'] = time.time() - start_time
    
    print(f"Assembly complete!")
    print(f"  Contigs generated: {len(contigs):,}")
    print(f"Time elapsed: {timing['assembly_time']:.2f} seconds")
    print()
    
    print("STEP 4: Calculating assembly statistics")
    print("-"*80)
    
    stats = dbg.get_assembly_stats()
    
    print(f"Assembly Statistics:")
    print(f"  Number of contigs:     {stats['num_contigs']:,}")
    print(f"  Total assembly length: {stats['total_length']:,} bp")
    print(f"  Longest contig:        {stats['longest_contig']:,} bp")
    print(f"  Shortest contig:       {stats['shortest_contig']:,} bp")
    print(f"  Mean contig length:    {stats['mean_length']:,.1f} bp")
    print(f"  N50:                   {stats['n50']:,} bp")
    print()
    
    # Display distribution of contig lengths
    contig_lengths = dbg.stats["contig_lengths"]
    
    print(f"Contig Length Distribution:")
    print(f"  Top 10 longest contigs:")
    for i, length in enumerate(contig_lengths[:10], 1):
        print(f"    {i:2d}. {length:,} bp")
    print()
    
    # Calculate coverage estimate
    genome_size_estimate = 2700000000  # Mouse genome ~2.7 Gbp
    coverage_estimate = (num_reads * avg_read_length) / genome_size_estimate
    assembly_fraction = (stats['total_length'] / genome_size_estimate) * 100
    
    print(f"Genome Coverage Analysis:")
    print(f"  Mouse genome size (expected): ~{genome_size_estimate:,} bp")
    print(f"  Estimated sequencing coverage: {coverage_estimate:.1f}x")
    print(f"  Assembly size vs. genome: {assembly_fraction:.1f}%")
    print()
    
    print("STEP 5: Writing output files")
    print("-"*80)
    
    # Write assembled contigs to FASTA
    dbg.write_fasta(output_fasta)
    print(f"✓ Contigs written to: {output_fasta}")
    
    # Write detailed statistics
    write_statistics_file(
        stats_file,
        input_file,
        num_reads,
        avg_read_length,
        k_mer_size,
        random_seed,
        num_nodes,
        num_edges,
        stats,
        contig_lengths,
        timing,
        coverage_estimate,
        assembly_fraction
    )
    print(f"✓ Statistics written to: {stats_file}")
    print()
    
    timing['total_time'] = (timing['read_time'] + timing['graph_time'] + 
                           timing['assembly_time'])
    
    print("="*80)
    print("ASSEMBLY COMPLETE")
    print("="*80)
    print(f"Total time: {timing['total_time']:.2f} seconds "
          f"({timing['total_time']/60:.2f} minutes)")
    print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print()
    
    print(f"Summary:")
    print(f"  • Processed {num_reads:,} reads ({total_bases:,} bp)")
    print(f"  • Built graph with {num_nodes:,} nodes and {num_edges:,} edges")
    print(f"  • Assembled {stats['num_contigs']:,} contigs")
    print(f"  • Total assembly: {stats['total_length']:,} bp (N50: {stats['n50']:,} bp)")
    print(f"  • Output files: {output_fasta}, {stats_file}")
    print()
    
    return {
        'dbg': dbg,
        'contigs': contigs,
        'stats': stats,
        'timing': timing,
        'graph_stats': {
            'num_nodes': num_nodes,
            'num_edges': num_edges,
            'avg_degree': avg_degree
        },
        'coverage': coverage_estimate
    }


print("\n" + "="*80)
print("MOUSE GENOME DE BRUIJN GRAPH ASSEMBLY")
print("Student Assignment Driver Program")
print("="*80 + "\n")

# Run the assembly pipeline
result = assemble_mouse_genome()

# Display sample contigs
print("="*80)
print("SAMPLE ASSEMBLED CONTIGS")
print("="*80 + "\n")

# create a generator object of contigs as strings (generator saves memory over a list here)
contigs = (tour_to_sequence(node_list) for node_list in result['contigs'])
i = 0    # initialize an index so we can track how many previews we've printed
for contig in contigs:
    preview = contig[:100] + "..." if len(contig) > 100 else contig
    print(f">contig_{i+1} length={len(contig)}")
    print(preview)
    print()

    # increment our index by 1
    i += 1
    if i == 4:
        # since we've printed contigs 1-5, we can stop iterating over our generator
        break

print("="*80)
print("✓ ASSEMBLY COMPLETE")
print("="*80)
print(f"\nResults saved to:")
print(f"  • mouse_assembly.fasta - {result['stats']['num_contigs']:,} assembled contigs")
print(f"  • mouse_assembly_stats.txt - Detailed assembly statistics")
print(f"\nKey metrics:")
print(f"  • Total assembly: {result['stats']['total_length']:,} bp")
print(f"  • N50: {result['stats']['n50']:,} bp")
print(f"  • Longest contig: {result['stats']['longest_contig']:,} bp")
print(f"  • Coverage: {result['coverage']:.1f}x")
print()


MOUSE GENOME DE BRUIJN GRAPH ASSEMBLY
Student Assignment Driver Program

MOUSE GENOME ASSEMBLY PIPELINE
Start time: 2026-02-25 00:33:23

STEP 1: Loading sequencing reads
--------------------------------------------------------------------------------
Input file: mouse_SE_150bp.fq.gz
Expected: 10 million reads, 150bp each

Time elapsed: 0.00 seconds
STEP 2: Building De Bruijn graph
--------------------------------------------------------------------------------
K-mer size: 51

[build] progress: 234056 reads

KeyboardInterrupt: 